# Notebook 04 · Same-Source Validation

Every predicted raster scored against **the product it was trained on** — CLMS
or GHS-BUILT-S in Milan, GHSL in Hanoi and HCMC — at that run's own held-out
test points.

This measures **agreement with the training target, not correctness**. A map
can match its training product closely and still be wrong about the ground.
Notebook 05 answers that separate question against the EarthLabel
photo-interpreted plots, and the two are never merged or compared directly.

Three techniques, the same three notebook 05 applies:

| | What it measures |
|---|---|
| **A** continuous | RMSE, MAE, bias, R² on the raw predicted % — no threshold, no binning |
| **B** hard confusion | overall accuracy and Cohen's κ at a 50 % impervious cut-off |
| **C** level confusion | 10-level confusion with quadratic weighted κ |

It reads existing outputs only: no Earth Engine, no retraining, no new
sampling. Each run already wrote its held-out test points with the training
target value attached, so the reference here is exactly what the model was
fitted against.

---

## Setup

In [1]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import rasterio
from sklearn.metrics import cohen_kappa_score, confusion_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

REPO    = Path.cwd()
OUT     = REPO / "output"
RESULTS = OUT / "validation_samesource"
RESULTS.mkdir(parents=True, exist_ok=True)

BIN_THRESHOLD = 50.0      # % impervious, the A/B split for technique B
N_LEVELS      = 10        # technique C bins: 0-10, 10-20, ... 90-100

print("results ->", RESULTS)

results -> D:\06_Polimi\2025-2026\IMD-Mapping\output\validation_samesource


---

## The runs, and the target each is scored against

In [2]:
# Every predicted raster, with the test points carrying its OWN training
# target. The pairing is the whole point: a CLMS-trained map is scored against
# CLMS, a GHSL-trained map against GHSL. Mixing them would not be same-source.
#
# `IMD` in each test-point file is the training-target value sampled at that
# point, written by the notebook that produced the run.

MILAN = OUT / "milan"
VIET  = OUT / "transfer_vietnam"

RUNS = []

# Milan -- four predictor sets x two training targets
for target, ras in (("clms", "IMD_predicted_RF_S2_Milan.tif"),
                    ("ghsl", "GHSL_predicted_RF_S2_Milan.tif")):
    for predictor in ("median", "stack", "percentile"):
        RUNS.append(dict(
            city="Milan", target=target.upper(), predictor=predictor,
            raster=MILAN / target / predictor / ras,
            points=MILAN / target / predictor / "spatial_test_pts.gpkg"))

for target, ras in (("clms", "IMD_predicted_RF_spatialCV2_Milan.tif"),
                    ("ghsl", "GHSL_predicted_RF_spatialCV2_Milan.tif")):
    RUNS.append(dict(
        city="Milan", target=target.upper(), predictor="embedding",
        raster=MILAN / target / "embedding" / ras,
        points=MILAN / target / "embedding" / "spatial_test_pts.gpkg"))

# Vietnam -- two predictors x two scenarios, all against GHSL
for city in ("Hanoi", "HCMC"):
    for predictor, suffix in (("embedding", ""), ("median", "_S2median")):
        for scenario in ("zeroshot", "localrf"):
            RUNS.append(dict(
                city=city, target="GHSL",
                predictor=f"{predictor}_{scenario}",
                raster=VIET / city.lower() / predictor
                       / f"IMD_{city}_10m_{scenario}{suffix}.tif",
                points=VIET / city.lower() / predictor
                       / f"samples_{city}_test.gpkg"))

print(f"{len(RUNS)} runs registered\n")
missing = 0
for r in RUNS:
    ok_r = "ok " if r["raster"].exists() else "MISS"
    ok_p = "ok " if r["points"].exists() else "MISS"
    if "MISS" in (ok_r, ok_p):
        missing += 1
        print(f"  {ok_r} raster  {ok_p} points   {r['city']:6s} "
              f"{r['target']:5s} {r['predictor']}")
print(f"{len(RUNS) - missing} / {len(RUNS)} runs have both raster and points")
assert missing == 0, f"{missing} run(s) incomplete -- see above"

16 runs registered

16 / 16 runs have both raster and points


---

## Sampling the predicted rasters at the held-out points

In [3]:
def sample_raster(raster_path, gdf):
    """Predicted value at each point centre, reprojected to the raster's CRS."""
    with rasterio.open(raster_path) as src:
        pts = gdf.to_crs(src.crs)
        coords = [(p.x, p.y) for p in pts.geometry]
        vals = np.array([v[0] for v in src.sample(coords)], dtype="float64")
        nodata = src.nodata
    if nodata is not None:
        vals[vals == nodata] = np.nan
    return vals


# The target column is named after the product: the CLMS runs write `IMD`,
# the GHSL runs write `GHSL`. Resolved rather than assumed, so a run whose
# points carry neither fails loudly instead of being scored against the
# wrong column.
TARGET_COL = {"CLMS": "IMD", "GHSL": "GHSL"}


def load_pair(run):
    """Observed (training target) and predicted % at the held-out test points."""
    gdf = gpd.read_file(run["points"])
    col = TARGET_COL[run["target"]]
    if col not in gdf.columns:
        # Vietnam transfer samples are GHSL-labelled but written by the
        # transfer notebooks, which kept the generic `IMD` name.
        col = "IMD" if "IMD" in gdf.columns else col
    if col not in gdf.columns:
        raise KeyError(
            f"{run['points']} has no target column; "
            f"expected {TARGET_COL[run['target']]!r} or 'IMD', "
            f"found {[c for c in gdf.columns if not c.startswith('A')]}")
    obs = gdf[col].to_numpy(dtype="float64")
    pred = sample_raster(run["raster"], gdf)
    keep = np.isfinite(obs) & np.isfinite(pred)
    return np.clip(obs[keep], 0, 100), np.clip(pred[keep], 0, 100)


print("sampling every run ...")
DATA = {}
for r in RUNS:
    key = (r["city"], r["target"], r["predictor"])
    DATA[key] = load_pair(r)
    n_all = len(gpd.read_file(r["points"]))
    n_ok = len(DATA[key][0])
    flag = "" if n_ok == n_all else f"   ({n_all - n_ok} dropped: nodata)"
    print(f"  {r['city']:6s} {r['target']:5s} {r['predictor']:20s} n={n_ok}{flag}")

sampling every run ...


  Milan  CLMS  median               n=1014


  Milan  CLMS  stack                n=1014


  Milan  CLMS  percentile           n=1014


  Milan  GHSL  median               n=998


...

  Hanoi  GHSL  median_localrf       n=895


  HCMC   GHSL  embedding_zeroshot   n=887


  HCMC   GHSL  embedding_localrf    n=887


  HCMC   GHSL  median_zeroshot      n=887


  HCMC   GHSL  median_localrf       n=887


---

## Technique A · continuous agreement

In [4]:
# ── Technique A · continuous agreement ───────────────────────────────────────
# No threshold and no binning: the predicted percentage against the training
# target percentage, exactly as both are stored.

def technique_a(obs, pred):
    err = pred - obs
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    return dict(
        n=len(obs),
        RMSE_pp=float(np.sqrt(np.mean(err ** 2))),
        MAE_pp=float(np.mean(np.abs(err))),
        bias_pp=float(np.mean(err)),
        R2=1 - ss_res / ss_tot if ss_tot else np.nan,
        pearson_r=float(np.corrcoef(obs, pred)[0, 1]) if len(obs) > 1 else np.nan,
    )


rows_a = [dict(city=c, target=t, predictor=p, **technique_a(*DATA[(c, t, p)]))
          for (c, t, p) in DATA]
tech_a = pd.DataFrame(rows_a)
tech_a.to_csv(RESULTS / "technique_A_continuous_metrics.csv", index=False)
print(tech_a.to_string(index=False))

 city target          predictor    n  RMSE_pp  MAE_pp  bias_pp     R2  pearson_r
Milan   CLMS             median 1014   11.285   7.820   -0.046  0.896      0.947
Milan   CLMS              stack 1014   10.864   7.641    0.169  0.904      0.951
Milan   CLMS         percentile 1014    9.462   6.389    0.104  0.927      0.963
Milan   GHSL             median  998   18.785  14.125   -0.386  0.720      0.848
Milan   GHSL              stack  998   18.288  13.793    0.140  0.734      0.857
Milan   GHSL         percentile  998   18.206  13.523   -0.341  0.737      0.859
Milan   CLMS          embedding 1014   14.123  10.675   -0.624  0.837      0.917
Milan   GHSL          embedding  998   18.361  13.820   -0.339  0.732      0.858
Hanoi   GHSL embedding_zeroshot  895   35.012  29.313   23.210 -0.014      0.676
Hanoi   GHSL  embedding_localrf  895    9.405   6.801    0.683  0.927      0.969
Hanoi   GHSL    median_zeroshot  895   37.504  29.182   24.068 -0.163      0.584
Hanoi   GHSL     median_loca

---

## Technique B · hard confusion at 50 %

In [5]:
# ── Technique B · hard confusion at 50 % ─────────────────────────────────────
# Both sides cut to pervious / impervious, then a two-class confusion matrix.
# Cohen's kappa rather than raw accuracy, so agreement expected by chance is
# discounted.

def technique_b(obs, pred, thr=BIN_THRESHOLD):
    o = (obs >= thr).astype(int)
    p = (pred >= thr).astype(int)
    cm = confusion_matrix(o, p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    def _div(a, b):
        return float(a / b) if b else np.nan
    return dict(
        n=len(obs),
        OA=_div(tn + tp, cm.sum()),
        kappa=float(cohen_kappa_score(o, p)) if len(set(o)) > 1 or len(set(p)) > 1 else np.nan,
        PA_pervious=_div(tn, tn + fp),
        PA_impervious=_div(tp, tp + fn),
        UA_pervious=_div(tn, tn + fn),
        UA_impervious=_div(tp, tp + fp),
        F1_impervious=_div(2 * tp, 2 * tp + fp + fn),
    )


rows_b = [dict(city=c, target=t, predictor=p, **technique_b(*DATA[(c, t, p)]))
          for (c, t, p) in DATA]
tech_b = pd.DataFrame(rows_b)
tech_b.to_csv(RESULTS / "technique_B_hard_confusion_metrics.csv", index=False)
print(tech_b.to_string(index=False))

 city target          predictor    n    OA  kappa  PA_pervious  PA_impervious  UA_pervious  UA_impervious  F1_impervious
Milan   CLMS             median 1014 0.934  0.867        0.938          0.931        0.920          0.946          0.938
Milan   CLMS              stack 1014 0.937  0.873        0.944          0.931        0.921          0.951          0.941
Milan   CLMS         percentile 1014 0.936  0.871        0.925          0.945        0.935          0.937          0.941
Milan   GHSL             median  998 0.831  0.661        0.817          0.844        0.839          0.823          0.833
Milan   GHSL              stack  998 0.842  0.683        0.833          0.850        0.847          0.837          0.844
Milan   GHSL         percentile  998 0.827  0.653        0.813          0.840        0.835          0.819          0.830
Milan   CLMS          embedding 1014 0.891  0.780        0.897          0.885        0.869          0.910          0.897
Milan   GHSL          embedding 

---

## Technique C · 10-level confusion

In [6]:
# ── Technique C · 10-level confusion ─────────────────────────────────────────
# No threshold: both sides binned into ten 10-point levels. Quadratic weighted
# kappa, so being two levels out is penalised more than being one level out --
# the ordering of the levels carries information a plain kappa throws away.

def to_level(v, n=N_LEVELS):
    return np.clip((np.asarray(v) // (100 / n)).astype(int), 0, n - 1)


def technique_c(obs, pred, n=N_LEVELS):
    lo, lp = to_level(obs, n), to_level(pred, n)
    exact = float(np.mean(lo == lp))
    within1 = float(np.mean(np.abs(lo - lp) <= 1))
    try:
        qwk = float(cohen_kappa_score(lo, lp, weights="quadratic",
                                      labels=list(range(n))))
    except ValueError:
        qwk = np.nan
    return dict(
        n=len(obs),
        exact_match_OA=exact,
        within_1_level_OA=within1,
        MAE_levels=float(np.mean(np.abs(lo - lp))),
        MAE_pp=float(np.mean(np.abs(pred - obs))),
        RMSE_pp=float(np.sqrt(np.mean((pred - obs) ** 2))),
        quad_weighted_kappa=qwk,
    )


rows_c = [dict(city=c, target=t, predictor=p, **technique_c(*DATA[(c, t, p)]))
          for (c, t, p) in DATA]
tech_c = pd.DataFrame(rows_c)
tech_c.to_csv(RESULTS / "technique_C_level_confusion_metrics.csv", index=False)
print(tech_c.to_string(index=False))

 city target          predictor    n  exact_match_OA  within_1_level_OA  MAE_levels  MAE_pp  RMSE_pp  quad_weighted_kappa
Milan   CLMS             median 1014           0.515              0.863       0.678   7.820   11.285                0.942
Milan   CLMS              stack 1014           0.515              0.877       0.653   7.641   10.864                0.946
Milan   CLMS         percentile 1014           0.577              0.903       0.549   6.389    9.462                0.957
Milan   GHSL             median  998           0.321              0.648       1.297  14.125   18.785                0.830
Milan   GHSL              stack  998           0.328              0.667       1.241  13.793   18.288                0.844
Milan   GHSL         percentile  998           0.344              0.648       1.257  13.523   18.206                0.837
Milan   CLMS          embedding 1014           0.373              0.754       0.969  10.675   14.123                0.900
Milan   GHSL          em

---

## All three techniques, per city

In [7]:
# ── One table per city, the three techniques side by side ────────────────────
# R2, kappa and quadratic weighted kappa are the three headline numbers, the
# same three the presentation quotes for the independent validation.

ranking = (tech_a[["city", "target", "predictor", "n", "R2", "RMSE_pp", "MAE_pp", "bias_pp"]]
           .merge(tech_b[["city", "target", "predictor", "kappa", "OA"]],
                  on=["city", "target", "predictor"])
           .merge(tech_c[["city", "target", "predictor", "quad_weighted_kappa"]],
                  on=["city", "target", "predictor"]))
ranking = ranking.sort_values(["city", "target", "R2"], ascending=[True, True, False])
ranking.to_csv(RESULTS / "ranking_all_runs.csv", index=False)

for city in ("Milan", "Hanoi", "HCMC"):
    sub = ranking[ranking.city == city]
    if len(sub):
        print(f"\n===== {city} — same-source validation =====")
        print(sub.drop(columns="city").to_string(index=False))


===== Milan — same-source validation =====
target  predictor    n    R2  RMSE_pp  MAE_pp  bias_pp  kappa    OA  quad_weighted_kappa
  CLMS percentile 1014 0.927    9.462   6.389    0.104  0.871 0.936                0.957
  CLMS      stack 1014 0.904   10.864   7.641    0.169  0.873 0.937                0.946
  CLMS     median 1014 0.896   11.285   7.820   -0.046  0.867 0.934                0.942
  CLMS  embedding 1014 0.837   14.123  10.675   -0.624  0.780 0.891                0.900
  GHSL percentile  998 0.737   18.206  13.523   -0.341  0.653 0.827                0.837
  GHSL      stack  998 0.734   18.288  13.793    0.140  0.683 0.842                0.844
  GHSL  embedding  998 0.732   18.361  13.820   -0.339  0.677 0.839                0.830
  GHSL     median  998 0.720   18.785  14.125   -0.386  0.661 0.831                0.830

===== Hanoi — same-source validation =====
target          predictor   n     R2  RMSE_pp  MAE_pp  bias_pp  kappa    OA  quad_weighted_kappa
  GHSL     med

---

## Figures

In [8]:
# ── Figure · the three techniques per run ────────────────────────────────────
# Panel titles name the technique and the city; the report supplies the figure
# title and caption.

matplotlib.rcParams.update({"font.size": 9, "axes.grid": True,
                            "grid.alpha": 0.3, "axes.axisbelow": True})

METRICS = [("R2", "R² · technique A (continuous)"),
           ("kappa", "κ · technique B (50 % confusion)"),
           ("quad_weighted_kappa", "QWK · technique C (10-level confusion)")]

for city in ("Milan", "Hanoi", "HCMC"):
    sub = ranking[ranking.city == city].copy()
    if not len(sub):
        continue
    sub["label"] = sub["target"] + " · " + sub["predictor"]
    sub = sub.sort_values("R2")
    fig, axes = plt.subplots(1, 3, figsize=(13, max(3, 0.32 * len(sub) + 1.6)),
                             sharey=True)
    for ax, (col, title) in zip(axes, METRICS):
        ax.barh(sub["label"], sub[col], color="#4878a8")
        ax.set_title(title, fontsize=9)
        ax.set_xlim(min(0, float(sub[col].min()) * 1.1), 1.0)
        for y, v in enumerate(sub[col]):
            ax.text(v, y, f" {v:.3f}", va="center", fontsize=7.5)
    axes[0].set_ylabel(f"{city} · training target · predictor")
    fig.tight_layout()
    out = RESULTS / f"fig_{city.lower()}_three_techniques.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("saved", out.name)

saved fig_milan_three_techniques.png
saved fig_hanoi_three_techniques.png


saved fig_hcmc_three_techniques.png


---

## Manifest

In [9]:
# ── Run manifest ─────────────────────────────────────────────────────────────
summary = dict(
    notebook="04_SameSource_Validation.ipynb",
    validation="same-source (vs the training target)",
    techniques=["A continuous", "B hard confusion at 50 %",
                "C 10-level confusion"],
    n_runs=len(RUNS),
    bin_threshold_pct=BIN_THRESHOLD,
    n_levels=N_LEVELS,
    runs=[{"city": c, "target": t, "predictor": p, "n": int(len(DATA[(c, t, p)][0]))}
          for (c, t, p) in DATA],
)
with open(RESULTS / "samesource_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)

print(f"\n-- wrote {len(list(RESULTS.iterdir()))} files to {RESULTS} --")
for f in sorted(RESULTS.iterdir()):
    print(f"   {f.name:46s} {f.stat().st_size / 1024:8.1f} kB")


-- wrote 8 files to D:\06_Polimi\2025-2026\IMD-Mapping\output\validation_samesource --
   fig_hanoi_three_techniques.png                     52.2 kB
   fig_hcmc_three_techniques.png                      53.4 kB
   fig_milan_three_techniques.png                     70.2 kB
   ranking_all_runs.csv                                2.6 kB
   samesource_summary.json                             2.1 kB
   technique_A_continuous_metrics.csv                  2.0 kB
   technique_B_hard_confusion_metrics.csv              2.6 kB
   technique_C_level_confusion_metrics.csv             2.3 kB
